# 🚀 Full RAG Pipeline — Hybrid Search + Reranking + Cache + Memory
All stages connected end-to-end.

In [1]:
!pip install numpy==1.26.4

  Using cached numpy-1.26.4-cp310-cp310-win_amd64.whl.metadata (61 kB)
Using cached numpy-1.26.4-cp310-cp310-win_amd64.whl (15.8 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 2.2.6
    Uninstalling numpy-2.2.6:
      Successfully uninstalled numpy-2.2.6


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.


In [2]:
!pip install redis cohere rank_bm25
%pip install -Uq "unstructured[all-docs]" langchain_chroma langchain langchain-community langchain-openai python_dotenv
!pip install langchain langchain-openai langchain-chroma chromadb unstructured pdf2image pillow python-dotenv "unstructured[pdf]"

Note: you may need to restart the kernel to use updated packages.


In [3]:
import os, json, hashlib, sqlite3
import numpy as np
import redis
import cohere
from collections import deque
from typing import List
from dotenv import load_dotenv

# Unstructured
import unstructured_client
from unstructured_client.models import operations, shared
try:
    from unstructured.staging.base import elements_from_dicts
except ImportError:
    from unstructured.staging.base import dict_to_elements as elements_from_dicts
from unstructured.chunking.title import chunk_by_title

# LangChain
from langchain_core.documents import Document
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.messages import HumanMessage
from rank_bm25 import BM25Okapi

load_dotenv(override=True)

# ── Shared singletons (created once, reused everywhere) ──────────────────────
embeddings  = OpenAIEmbeddings(model="text-embedding-3-small")
fast_llm    = ChatOpenAI(model="gpt-4o-mini", temperature=0,streaming=True)
answer_llm  = ChatOpenAI(model="gpt-4o-mini",      temperature=0,streaming=True)
co          = cohere.Client(os.getenv("COHERE_API_KEY"))
r           = redis.from_url(os.getenv("REDIS_URL"), decode_responses=True)

# Quick connectivity checks
r.set("test", "hello from notebook"); print(r.get("test"))
print("All clients initialised")

c:\Users\Garima\anaconda\envs\multimodal_rag\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


hello from notebook
All clients initialised


## Stage 1 — Document Partitioning (Unstructured.io)

In [4]:

unstructured_client_obj = unstructured_client.UnstructuredClient(
    api_key_auth=os.getenv("UNSTRUCTURED_API_KEY"),
)

def partition_document(file_path: str):
    """Extract elements from PDF using Unstructured.io API"""
    print(f"📄 Partitioning document: {file_path}")
    with open(file_path, "rb") as f:
        file_bytes = f.read()
    req = operations.PartitionRequest(
        partition_parameters=shared.PartitionParameters(
            files=shared.Files(content=file_bytes, file_name=file_path),
            strategy=shared.Strategy.HI_RES,
            split_pdf_page=True,
            split_pdf_concurrency_level=10,
            extract_image_block_types=["Image"],
        ),
    )
    resp = unstructured_client_obj.general.partition(request=req)
    elements = elements_from_dicts(resp.elements)
    print(f"Extracted {len(elements)} elements")
    return elements

## Stage 2 — Chunking

In [ ]:

def create_chunks_by_title(elements):
    """Create intelligent chunks using title-based strategy"""
    print(" Creating smart chunks...")
    chunks = chunk_by_title(
        elements,
        max_characters=3000,
        new_after_n_chars=2400,
        combine_text_under_n_chars=500
    )
    print(f"Created {len(chunks)} chunks")
    return chunks

## Stage 3 — AI Summarisation

In [ ]:

def separate_content_types(chunk):
    content_data = {'text': chunk.text, 'tables': [], 'images': [], 'types': ['text']}
    if hasattr(chunk, 'metadata') and hasattr(chunk.metadata, 'orig_elements'):
        for element in chunk.metadata.orig_elements:
            element_type = type(element).__name__
            if element_type == 'Table':
                content_data['types'].append('table')
                content_data['tables'].append(getattr(element.metadata, 'text_as_html', element.text))
            elif element_type == 'Image':
                if hasattr(element, 'metadata') and hasattr(element.metadata, 'image_base64'):
                    content_data['types'].append('image')
                    content_data['images'].append(element.metadata.image_base64)
    content_data['types'] = list(set(content_data['types']))
    return content_data

def create_ai_enhanced_summary(text: str, tables: List[str], images: List[str]) -> str:
    try:
        prompt_text = f"""You are creating a searchable description for document content retrieval.

        CONTENT TO ANALYZE:
        TEXT CONTENT:
        {text}

        """
        
        # Add tables if present
        if tables:
            prompt_text += "TABLES:\n"
            for i, table in enumerate(tables):
                prompt_text += f"Table {i+1}:\n{table}\n\n"
        
        #  ALWAYS add task instructions (moved outside if block)
        prompt_text += """
YOUR TASK:
Generate a comprehensive, searchable description that covers:
1. Key facts, numbers, and data points from text and tables
2. Main topics and concepts discussed
3. Questions this content could answer
4. Visual content analysis (charts, diagrams, patterns in images)
5. Alternative search terms users might use
Make it detailed and searchable - prioritize findability over brevity.

SEARCHABLE DESCRIPTION:"""
        
        message_content = [{"type": "text", "text": prompt_text}]
        for image_base64 in images:
            message_content.append({
                "type": "image_url", 
                "image_url": {"url": f"data:image/jpeg;base64,{image_base64}"}
            })
        
        response = fast_llm.invoke([HumanMessage(content=message_content)])
        return response.content
        
    except Exception as e:
        print(f"      AI summary failed: {e}")
        summary = f"{text[:300]}..."
        if tables: summary += f" [Contains {len(tables)} table(s)]"
        if images: summary += f" [Contains {len(images)} image(s)]"
        return summary

def summarise_chunks(chunks):
    print(" Processing chunks with AI Summaries...")
    langchain_documents = []
    for i, chunk in enumerate(chunks):
        print(f"   Processing chunk {i+1}/{len(chunks)}")
        content_data = separate_content_types(chunk)
        print(f"     Types found: {content_data['types']}")
        print(f"     Tables: {len(content_data['tables'])}, Images: {len(content_data['images'])}")
        if content_data['tables'] or content_data['images']:
            print(f"     → Creating AI summary for mixed content...")
            try:
                enhanced_content = create_ai_enhanced_summary(content_data['text'], content_data['tables'], content_data['images'])
                print(f"     → AI summary created successfully")
                print(f"     → Enhanced content preview: {enhanced_content[:200]}...")
            except Exception as e:
                print(f"      AI summary failed: {e}")
                enhanced_content = content_data['text']
        else:
            print(f"     → Using raw text (no tables/images)")
            enhanced_content = content_data['text']
        doc = Document(
            page_content=enhanced_content,
            metadata={"original_content": json.dumps({
                "raw_text": content_data['text'],
                "tables_html": content_data['tables'],
                "images_base64": content_data['images']
            })}
        )
        langchain_documents.append(doc)
    print(f"Processed {len(langchain_documents)} chunks")
    return langchain_documents

## Stage 4 — Vector Store (ChromaDB)

In [7]:

def create_vector_store(documents, persist_directory="dbv2/chroma_db"):
    print("🔮 Creating embeddings and storing in ChromaDB...")
    vectorstore = Chroma.from_documents(
        documents=documents,
        embedding=embeddings,
        persist_directory=persist_directory,
        collection_metadata={"hnsw:space": "cosine"}
    )
    print(f"Vector store created and saved to {persist_directory}")
    return vectorstore

def export_chunks_to_json(chunks, filename="chunks_export.json"):
    """Export processed chunks to clean JSON format"""
    export_data = []
    for i, doc in enumerate(chunks):
        chunk_data = {
            "chunk_id": i + 1,
            "enhanced_content": doc.page_content,
            "metadata": {"original_content": json.loads(doc.metadata.get("original_content", "{}"))}
        }
        export_data.append(chunk_data)
    with open(filename, 'w', encoding='utf-8') as f:
        json.dump(export_data, f, indent=2, ensure_ascii=False)
    print(f" Exported {len(export_data)} chunks to {filename}")
    return export_data

## Stage 5 — Hybrid Retriever (Dense + BM25)

In [ ]:

class HybridRetriever:
    """Chroma (dense) + BM25 (lexical). Reuses query vector to avoid re-embedding."""

    def __init__(self, vectordb, documents, alpha=0.6):
        self.vectordb = vectordb
        self.docs = documents
        self.alpha = alpha
        print(" Building BM25 index...")
        tokenized = [d.page_content.lower().split() for d in documents]
        self.bm25 = BM25Okapi(tokenized)
        print(f"Hybrid retriever ready ({len(documents)} docs)")

    def search(self, query, k=10, query_vec=None):
        if query_vec is not None:
            qv_list = query_vec.tolist() if hasattr(query_vec, "tolist") else list(query_vec)
            results = self.vectordb._collection.query(query_embeddings=[qv_list], n_results=len(self.docs))
            dense_scores = np.zeros(len(self.docs))
            for text, dist in zip(results["documents"][0], results["distances"][0]):
                for i, d in enumerate(self.docs):
                    if d.page_content == text:
                        dense_scores[i] = -dist
                        break
        else:
            dense_hits = self.vectordb.similarity_search_with_score(query, k=len(self.docs))
            dense_scores = np.zeros(len(self.docs))
            for doc, distance in dense_hits:
                for i, d in enumerate(self.docs):
                    if d.page_content == doc.page_content:
                        dense_scores[i] = -distance
                        break
        bm_scores = np.array(self.bm25.get_scores(query.lower().split()))
        def norm(x):
            return (x - x.min()) / (np.ptp(x) + 1e-9) if np.ptp(x) > 0 else x * 0
        fused = self.alpha * norm(dense_scores) + (1 - self.alpha) * norm(bm_scores)
        top_idx = np.argsort(-fused)[:k]
        return [self.docs[i] for i in top_idx]

## Stage 6 — Cohere Reranker

In [9]:

def llm_rerank(query, docs, top_k=3):
    """Cohere Rerank v3 — purpose-built reranker."""
    if len(docs) <= top_k:
        return docs
    texts = [d.page_content for d in docs]
    response = co.rerank(
        query=query,
        documents=texts,
        top_n=top_k,
        model="rerank-english-v3.0",
    )
    return [docs[r.index] for r in response.results]

print(" Cohere reranker ready")

 Cohere reranker ready


## Stage 7 — Semantic Cache (Redis)

In [10]:

class SemanticCache:
    """Redis cache — accepts pre-computed query vectors to avoid re-embedding."""

    def __init__(self, threshold=0.85, ttl_seconds=86400):
        self.threshold = threshold
        self.ttl = ttl_seconds
        self.redis = r  # reuse shared redis client
        self.embedder = embeddings
        try:
            self.redis.ping()
            print("Redis cache connected")
        except Exception as e:
            print(f" Redis failed: {e}")

    def _exact_key(self, query):
        return f"cache:exact:{hashlib.md5(query.encode()).hexdigest()}"

    def get(self, query, query_vec=None):
        exact = self.redis.get(self._exact_key(query))
        if exact:
            print("exact cache hit")
            return json.loads(exact)["answer"]
        qv = np.array(query_vec) if query_vec is not None else np.array(self.embedder.embed_query(query))
        for key in self.redis.scan_iter("cache:sem:*"):
            entry_json = self.redis.get(key)
            if not entry_json:
                continue
            entry = json.loads(entry_json)
            kv = np.array(entry["vector"])
            sim = np.dot(kv, qv) / (np.linalg.norm(kv) * np.linalg.norm(qv) + 1e-9)
            if sim >= self.threshold:
                print(f"semantic cache hit (sim={sim:.3f})")
                return entry["answer"]
        return None

    def put(self, query, answer, query_vec=None):
        qv = query_vec if query_vec is not None else self.embedder.embed_query(query)
        qv_list = list(map(float, qv))
        self.redis.setex(self._exact_key(query), self.ttl, json.dumps({"answer": answer}))
        sem_key = f"cache:sem:{hashlib.md5(query.encode()).hexdigest()}"
        self.redis.setex(sem_key, self.ttl, json.dumps({"answer": answer, "vector": qv_list, "query": query}))

    def clear(self):
        for key in self.redis.scan_iter("cache:*"):
            self.redis.delete(key)
        print("Cache cleared")


cache = SemanticCache(threshold=0.85, ttl_seconds=86400)

Redis cache connected


## Stage 8 — Memory (SQLite)

In [ ]:

class Memory:
    """3-tier memory with SQLite persistence."""

    def __init__(self, db_path="memory.db", session_id="default", short_size=4):
        self.session_id = session_id
        self.short_size = short_size
        self.conn = sqlite3.connect(db_path, check_same_thread=False)
        self._init_db()
        self.short = deque(self._load_recent(), maxlen=short_size)
        self.summary = self._load_summary()
        print(f"Memory loaded: {len(self.short)} turns, session='{session_id}'")

    def _init_db(self):
        self.conn.executescript("""
            CREATE TABLE IF NOT EXISTS turns (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                session_id TEXT NOT NULL,
                user_msg TEXT NOT NULL,
                assistant_msg TEXT NOT NULL,
                embedding BLOB NOT NULL,
                created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
            );
            CREATE INDEX IF NOT EXISTS idx_session ON turns(session_id);
            CREATE TABLE IF NOT EXISTS summaries (
                session_id TEXT PRIMARY KEY,
                summary TEXT,
                updated_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
            );
        """)
        self.conn.commit()

    def _load_recent(self):
        rows = self.conn.execute(
            "SELECT user_msg, assistant_msg FROM turns WHERE session_id = ? ORDER BY id DESC LIMIT ?",
            (self.session_id, self.short_size)
        ).fetchall()
        return [(u, a) for u, a in reversed(rows)]

    def _load_summary(self):
        row = self.conn.execute(
            "SELECT summary FROM summaries WHERE session_id = ?", (self.session_id,)
        ).fetchone()
        return row[0] if row else ""

    def add_turn(self, user_msg, assistant_msg):
        turn_text = f"User: {user_msg}\nAssistant: {assistant_msg}"
        vec = np.array(embeddings.embed_query(turn_text), dtype=np.float32)
        self.conn.execute(
            "INSERT INTO turns (session_id, user_msg, assistant_msg, embedding) VALUES (?, ?, ?, ?)",
            (self.session_id, user_msg, assistant_msg, vec.tobytes())
        )
        self.conn.commit()
        if len(self.short) == self.short.maxlen:
            old_u, old_a = self.short[0]
            self.summary = fast_llm.invoke(
                f"Prior summary: {self.summary or '(none)'}\n\nNew turn:\nUser: {old_u}\nAssistant: {old_a}\n\n"
                "Update the summary in under 80 words, keeping key facts:"
            ).content
            self.conn.execute(
                "INSERT INTO summaries (session_id, summary) VALUES (?, ?) "
                "ON CONFLICT(session_id) DO UPDATE SET summary=excluded.summary, updated_at=CURRENT_TIMESTAMP",
                (self.session_id, self.summary)
            )
            self.conn.commit()
        self.short.append((user_msg, assistant_msg))

    def context(self, query, query_vec=None):
        parts = []
        if self.summary:
            parts.append(f"[Summary so far]\n{self.summary}")
        qv = np.array(query_vec, dtype=np.float32) if query_vec is not None else \
             np.array(embeddings.embed_query(query), dtype=np.float32)
        rows = self.conn.execute(
            "SELECT user_msg, assistant_msg, embedding FROM turns WHERE session_id = ? ORDER BY id DESC",
            (self.session_id,)
        ).fetchall()
        if len(rows) > self.short_size:
            older = rows[self.short_size:]
            scored = []
            for u, a, emb_bytes in older:
                v = np.frombuffer(emb_bytes, dtype=np.float32)
                sim = np.dot(v, qv) / (np.linalg.norm(v) * np.linalg.norm(qv) + 1e-9)
                scored.append((sim, u, a))
            scored.sort(key=lambda x: -x[0])
            for sim, u, a in scored[:2]:
                parts.append(f"[Relevant past turn]\nUser: {u}\nAssistant: {a}")
        for u, a in self.short:
            parts.append(f"User: {u}\nAssistant: {a}")
        return "\n\n".join(parts)

    def clear(self):
        self.conn.execute("DELETE FROM turns WHERE session_id = ?", (self.session_id,))
        self.conn.execute("DELETE FROM summaries WHERE session_id = ?", (self.session_id,))
        self.conn.commit()
        self.short.clear()
        self.summary = ""
        print(f"Cleared session '{self.session_id}'")


memory = Memory(db_path="memory.db", session_id="user_garima", short_size=4)

Memory loaded: 4 turns, session='user_garima'


## Stage 9 — Answer Generation

In [12]:
def generate_final_answer_stream(chunks, query):
    """Streaming version of multimodal answer generation"""
    try:
        prompt_text = f"""Based on the following documents, please answer this question: {query}\n\nCONTENT TO ANALYZE:\n"""

        for i, chunk in enumerate(chunks):
            prompt_text += f"--- Document {i+1} ---\n"

            if "original_content" in chunk.metadata:
                original_data = json.loads(chunk.metadata["original_content"])

                raw_text = original_data.get("raw_text", "")
                if raw_text:
                    prompt_text += f"TEXT:\n{raw_text}\n\n"

                tables_html = original_data.get("tables_html", [])
                if tables_html:
                    prompt_text += "TABLES:\n"
                    for j, table in enumerate(tables_html):
                        prompt_text += f"Table {j+1}:\n{table}\n\n"

            prompt_text += "\n"

        prompt_text += """\nPlease provide a clear, comprehensive answer using the text, tables, and images above. If the documents don't contain sufficient information to answer the question, say "I don't have enough information to answer that question based on the provided documents."\n\nANSWER:"""

        message_content = [{"type": "text", "text": prompt_text}]

        for chunk in chunks:
            if "original_content" in chunk.metadata:
                original_data = json.loads(chunk.metadata["original_content"])

                for image_base64 in original_data.get("images_base64", []):
                    message_content.append({
                        "type": "image_url",
                        "image_url": {"url": f"data:image/jpeg;base64,{image_base64}"}
                    })

        # STREAMING PART 
        stream = answer_llm.stream([HumanMessage(content=message_content)])

        for chunk in stream:
            if hasattr(chunk, "content") and chunk.content:
                yield chunk.content

    except Exception as e:
        print(f"Answer generation failed: {e}")
        yield "Sorry, I encountered an error while generating the answer."

## 🔗 Full Ingestion Pipeline

In [13]:
def run_complete_ingestion_pipeline(pdf_path: str):
    """Run the complete RAG ingestion pipeline"""
    print("Starting RAG Ingestion Pipeline")
    print("=" * 50)
    elements         = partition_document(pdf_path)           # Stage 1
    raw_chunks       = create_chunks_by_title(elements)       # Stage 2
    summarised_chunks = summarise_chunks(raw_chunks)          # Stage 3
    db               = create_vector_store(summarised_chunks, persist_directory="dbv2/chroma_db")  # Stage 4
    print("Pipeline completed successfully!")
    return db, summarised_chunks

# Clear old database
import shutil
shutil.rmtree("dbv2/chroma_db", ignore_errors=True)
print("Old database deleted")

# Run ingestion
db, processed_chunks = run_complete_ingestion_pipeline("Vein_Recognition_Documentation (13).pdf")


# Clear cache 
cache.clear()
print("Cache cleared")

# Clear memory 
memory.clear()
print("Memory cleared")


# ▶ Initialise hybrid retriever now that we have db + chunks
hybrid = HybridRetriever(db, processed_chunks, alpha=0.6)

Old database deleted
Starting RAG Ingestion Pipeline
📄 Partitioning document: Vein_Recognition_Documentation (13).pdf


INFO: HTTP Request: GET https://api.unstructuredapp.io/general/docs "HTTP/1.1 200 OK"
INFO: HTTP Request: POST https://api.unstructuredapp.io/general/v0/general "HTTP/1.1 200 OK"
INFO: HTTP Request: POST https://api.unstructuredapp.io/general/v0/general "HTTP/1.1 200 OK"
INFO: HTTP Request: POST https://api.unstructuredapp.io/general/v0/general "HTTP/1.1 200 OK"
INFO: HTTP Request: POST https://api.unstructuredapp.io/general/v0/general "HTTP/1.1 200 OK"
INFO: HTTP Request: POST https://api.unstructuredapp.io/general/v0/general "HTTP/1.1 200 OK"
INFO: HTTP Request: POST https://api.unstructuredapp.io/general/v0/general "HTTP/1.1 200 OK"
INFO: HTTP Request: POST https://api.unstructuredapp.io/general/v0/general "HTTP/1.1 200 OK"


Extracted 190 elements
🔨 Creating smart chunks...
Created 28 chunks
🧠 Processing chunks with AI Summaries...
   Processing chunk 1/28
     Types found: ['text']
     Tables: 0, Images: 0
     → Using raw text (no tables/images)
   Processing chunk 2/28
     Types found: ['text']
     Tables: 0, Images: 0
     → Using raw text (no tables/images)
   Processing chunk 3/28
     Types found: ['text', 'image']
     Tables: 0, Images: 1
     → Creating AI summary for mixed content...


INFO: HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


     → AI summary created successfully
     → Enhanced content preview: ### SEARCHABLE DESCRIPTION

**Key Facts, Numbers, and Data Points:**
- **Biometric Categories:** Two main groups: physiological (e.g., fingerprint, face, iris, ear, dorsal hand vein) and behavioral (e...
   Processing chunk 4/28
     Types found: ['text']
     Tables: 0, Images: 0
     → Using raw text (no tables/images)
   Processing chunk 5/28
     Types found: ['text']
     Tables: 0, Images: 0
     → Using raw text (no tables/images)
   Processing chunk 6/28
     Types found: ['text']
     Tables: 0, Images: 0
     → Using raw text (no tables/images)
   Processing chunk 7/28
     Types found: ['text', 'image']
     Tables: 0, Images: 1
     → Creating AI summary for mixed content...


...

     → AI summary created successfully
     → Enhanced content preview: ### SEARCHABLE DESCRIPTION:

#### Key Facts, Numbers, and Data Points:
1. **Research Articles**:
   - **K. Shaheed et al. (2022)**: Explores finger-vein presentation attack detection using depthwise s...
   Processing chunk 27/28
     Types found: ['text', 'image', 'table']
     Tables: 1, Images: 1
     → Creating AI summary for mixed content...


INFO: HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


     → AI summary created successfully
     → Enhanced content preview: ### SEARCHABLE DESCRIPTION:

**Key Facts, Numbers, and Data Points:**
- **Figures Analyzed:**
  - **Fig. 12:** Performance comparison of Ensemble Neural Network (NN) with and without U-Net on Dataset ...
   Processing chunk 28/28
     Types found: ['text']
     Tables: 0, Images: 0
     → Using raw text (no tables/images)
Processed 28 chunks
🔮 Creating embeddings and storing in ChromaDB...


INFO: HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


Vector store created and saved to dbv2/chroma_db
Pipeline completed successfully!
Cache cleared
Cache cleared
Cleared session 'user_garima'
Memory cleared
🔤 Building BM25 index...
Hybrid retriever ready (28 docs)


## 🔗 Full Query Pipeline (Hybrid → Rerank → Cache → Memory → Answer)

In [ ]:
def rag_query(query: str, k_hybrid: int = 10, k_rerank: int = 3) -> str:
    """
    Full pipeline:
      1. Embed query ONCE   
      2. Check semantic cache  → return early if hit
      3. Hybrid retrieval      (dense + BM25) using pre-computed vector
      4. Cohere rerank
      5. Build prompt with memory context
      6. Generate answer
      7. Store in cache + memory
    """
    print(f"\n{'='*60}")
    print(f"Query: {query}")
    print(f"{'='*60}")

    # ── Step 1: Embed query ONCE ─────────────────────────────────
    query_vec = np.array(embeddings.embed_query(query))

    # ── Step 2: Cache check ──────────────────────────────────────
    cached = cache.get(query, query_vec=query_vec)
    if cached:
        print(" Returning cached answer")
        print(cached)
        return cached

    # ── Step 3: Hybrid retrieval ─────────────────────────────────
    print(f" Hybrid search (k={k_hybrid})...")
    retrieved = hybrid.search(query, k=k_hybrid, query_vec=query_vec)
    print(f"   Retrieved {len(retrieved)} docs")

    # ── Step 4: Cohere rerank ────────────────────────────────────
    print(f" Reranking to top {k_rerank}...")
    reranked = llm_rerank(query, retrieved, top_k=k_rerank)
    print(f"   Reranked to {len(reranked)} docs")

    # ── Step 5: Memory context ───────────────────────────────────
    mem_ctx = memory.context(query, query_vec=query_vec)
    if mem_ctx:
        # Prepend memory as an extra doc
        mem_doc = Document(page_content=f"[Conversation History]\n{mem_ctx}", metadata={"original_content": json.dumps({"raw_text": mem_ctx, "tables_html": [], "images_base64": []})})
        reranked = [mem_doc] + reranked

    # ── Step 6: Generate answer ──────────────────────────────────
    print("💬 Generating answer (streaming)...")

    answer = ""
    for chunk in generate_final_answer_stream(reranked, query):
        print(chunk, end="", flush=True)
        answer += chunk

    # ── Step 7: Store in cache + memory ─────────────────────────
    cache.put(query, answer, query_vec=query_vec)
    memory.add_turn(query, answer)

    print("\nAnswer:")
    print(answer)
    return answer


print("rag_query() ready — all stages connected")

rag_query() ready — all stages connected


## ▶ Run a Query

In [19]:
answer = rag_query("Explain the dorsal venous network")


Query: Explain the dorsal venous network


INFO: HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


exact cache hit
⚡ Returning cached answer
The dorsal venous network refers to the system of veins located on the back of the hand. It primarily consists of the dorsal venous arch, along with the basilic and cephalic veins. This network plays a crucial role in venous return from the hand and is significant in various medical and biometric applications.

In the context of biometric identification, the dorsal hand vein (DHV) recognition system utilizes the unique patterns of veins beneath the skin. This method is advantageous because vein patterns are difficult to replicate or alter, providing a high level of security for personal identification. The process typically involves using infrared light to scan the veins, which appear as dark lines due to the absorption of light by hemoglobin in the blood. The acquired images are then processed to enhance the visibility of the veins for accurate recognition.

Overall, the dorsal venous network is not only vital for physiological functions but a

In [18]:

answer2 = rag_query("What CNN models were used for classification?")


Query: What CNN models were used for classification?


INFO: HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


🔎 Hybrid search (k=10)...
   Retrieved 10 docs
🏆 Reranking to top 3...


INFO: HTTP Request: POST https://api.cohere.com/v1/rerank "HTTP/1.1 200 OK"


   Reranked to 3 docs
💬 Generating answer (streaming)...


INFO: HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


The CNN models used for classification in the dorsal hand vein recognition system are:

1. **VGG16**
2. **ResNet50**
3. **MobileNet**
4. **InceptionV3**

These models are utilized individually and in ensemble combinations to enhance recognition accuracy.

INFO: HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"



Answer:
The CNN models used for classification in the dorsal hand vein recognition system are:

1. **VGG16**
2. **ResNet50**
3. **MobileNet**
4. **InceptionV3**

These models are utilized individually and in ensemble combinations to enhance recognition accuracy.


In [20]:
answer3 = rag_query("Who is the author")


Query: Who is the author


INFO: HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


🔎 Hybrid search (k=10)...
   Retrieved 10 docs
🏆 Reranking to top 3...


INFO: HTTP Request: POST https://api.cohere.com/v1/rerank "HTTP/1.1 200 OK"


   Reranked to 3 docs
💬 Generating answer (streaming)...


INFO: HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


The author of the document is Garima Agarwal, along with co-authors Debbrota Paul Chowdhury and Sambit Bakshi, as indicated in Document 3.

INFO: HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"



Answer:
The author of the document is Garima Agarwal, along with co-authors Debbrota Paul Chowdhury and Sambit Bakshi, as indicated in Document 3.


In [21]:
answer3 = rag_query("what all the preprocessing step were used")


Query: what all the preprocessing step were used


INFO: HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


🔎 Hybrid search (k=10)...
   Retrieved 10 docs
🏆 Reranking to top 3...


INFO: HTTP Request: POST https://api.cohere.com/v1/rerank "HTTP/1.1 200 OK"


   Reranked to 3 docs
💬 Generating answer (streaming)...


INFO: HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


The preprocessing steps used for the dorsal hand vein recognition system, as detailed in the provided documents, include the following:

1. **Image Resizing**: All images were resized to 224 × 224 pixels to maintain uniform dimensions across the dataset.

2. **Contrast Enhancement**:
   - **CLAHE (Contrast Limited Adaptive Histogram Equalization)**: This technique was applied to enhance the visibility of vein structures by adapting local contrast while minimizing noise amplification. A clip limit of 1.5 and a tile grid size of 8 × 8 were used.

3. **Thresholding**:
   - **Adaptive Thresholding**: The Gaussian method was employed with a block size of 21 and a constant value of C = 2 to dynamically adjust the threshold based on local pixel intensity variations.

4. **Morphological Operations**:
   - **Dilation**: This operation was performed using a 3 × 3 kernel to reconnect broken vein segments and strengthen continuity in vein patterns.

5. **Edge Detection**:
   - **Canny Edge Detecti

INFO: HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"



Answer:
The preprocessing steps used for the dorsal hand vein recognition system, as detailed in the provided documents, include the following:

1. **Image Resizing**: All images were resized to 224 × 224 pixels to maintain uniform dimensions across the dataset.

2. **Contrast Enhancement**:
   - **CLAHE (Contrast Limited Adaptive Histogram Equalization)**: This technique was applied to enhance the visibility of vein structures by adapting local contrast while minimizing noise amplification. A clip limit of 1.5 and a tile grid size of 8 × 8 were used.

3. **Thresholding**:
   - **Adaptive Thresholding**: The Gaussian method was employed with a block size of 21 and a constant value of C = 2 to dynamically adjust the threshold based on local pixel intensity variations.

4. **Morphological Operations**:
   - **Dilation**: This operation was performed using a 3 × 3 kernel to reconnect broken vein segments and strengthen continuity in vein patterns.

5. **Edge Detection**:
   - **Canny Edg

In [22]:
answer4 = rag_query("step wise step expalin me")


Query: step wise step expalin me


INFO: HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


🔎 Hybrid search (k=10)...
   Retrieved 10 docs
🏆 Reranking to top 3...


INFO: HTTP Request: POST https://api.cohere.com/v1/rerank "HTTP/1.1 200 OK"


   Reranked to 3 docs
💬 Generating answer (streaming)...


INFO: HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
INFO: Retrying request to /chat/completions in 0.718000 seconds
INFO: HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


To explain the dorsal venous network and its relevance in biometric identification, we can break down the information step by step:

### Step 1: Understanding the Dorsal Venous Network
- **Definition**: The dorsal venous network is a system of veins located on the back of the hand. It includes the dorsal venous arch, basilic vein, and cephalic vein.
- **Function**: This network is crucial for the venous return from the hand, which is essential for maintaining proper blood circulation.

### Step 2: Biometric Identification Using Vein Patterns
- **Dorsal Hand Vein (DHV) Recognition**: This biometric system utilizes the unique patterns of veins beneath the skin for identification.
- **Advantages**: Vein patterns are difficult to replicate or alter, providing a high level of security for personal identification.

### Step 3: Imaging Process
- **Infrared Scanning**: The process typically involves using infrared light to scan the veins. The veins appear as dark lines due to the absorption of

INFO: HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO: HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"



Answer:
To explain the dorsal venous network and its relevance in biometric identification, we can break down the information step by step:

### Step 1: Understanding the Dorsal Venous Network
- **Definition**: The dorsal venous network is a system of veins located on the back of the hand. It includes the dorsal venous arch, basilic vein, and cephalic vein.
- **Function**: This network is crucial for the venous return from the hand, which is essential for maintaining proper blood circulation.

### Step 2: Biometric Identification Using Vein Patterns
- **Dorsal Hand Vein (DHV) Recognition**: This biometric system utilizes the unique patterns of veins beneath the skin for identification.
- **Advantages**: Vein patterns are difficult to replicate or alter, providing a high level of security for personal identification.

### Step 3: Imaging Process
- **Infrared Scanning**: The process typically involves using infrared light to scan the veins. The veins appear as dark lines due to the abso

In [23]:
answer4 = rag_query("step wise step explain me the preprocessing step")


Query: step wise step explain me the preprocessing step


INFO: HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


🔎 Hybrid search (k=10)...
   Retrieved 10 docs
🏆 Reranking to top 3...


INFO: HTTP Request: POST https://api.cohere.com/v1/rerank "HTTP/1.1 200 OK"


   Reranked to 3 docs
💬 Generating answer (streaming)...


INFO: HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


To explain the preprocessing steps used in the dorsal hand vein recognition system, we can break it down into a clear, step-by-step process:

### Step 1: Image Loading
- **Load Image**: The first step involves loading the original image of the dorsal hand veins.

### Step 2: Image Resizing
- **Resize Images**: All images are resized to a uniform dimension of 224 × 224 pixels to ensure consistency across the dataset.

### Step 3: Contrast Enhancement
- **Apply CLAHE (Contrast Limited Adaptive Histogram Equalization)**: This technique enhances the visibility of vein structures by improving local contrast while minimizing noise. A clip limit of 1.5 and a tile grid size of 8 × 8 are used to balance enhancement and prevent artifacts.

### Step 4: Thresholding
- **Adaptive Thresholding**: The Gaussian method is applied with a block size of 21 and a constant value of C = 2. This dynamically adjusts the threshold based on local pixel intensity variations, ensuring clear separation of veins fro

INFO: HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO: HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
INFO: Retrying request to /chat/completions in 0.067000 seconds
INFO: HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
INFO: Retrying request to /chat/completions in 0.067000 seconds
INFO: HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 429 Too Many Requests"


RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for gpt-4o-mini in organization org-YClA9WuunoUdYJVFe2IwQzuG on tokens per min (TPM): Limit 200000, Used 200000, Requested 226. Please try again in 67ms. Visit https://platform.openai.com/account/rate-limits to learn more.', 'type': 'tokens', 'param': None, 'code': 'rate_limit_exceeded'}}

In [24]:
answer6 = rag_query("explain the result of this resesrch paper")



Query: explain the result of this resesrch paper


INFO: HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


🔎 Hybrid search (k=10)...
   Retrieved 10 docs
🏆 Reranking to top 3...


INFO: HTTP Request: POST https://api.cohere.com/v1/rerank "HTTP/1.1 200 OK"


   Reranked to 3 docs
💬 Generating answer (streaming)...


INFO: HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


The research paper presents a dorsal hand vein recognition system that utilizes two publicly available datasets to evaluate its effectiveness. The key findings and results of the study are as follows:

1. **Datasets Used**: The study employs two datasets:
   - **Dataset I**: The Dorsal Hand Veins Image Database, which includes 1,104 images from 138 individuals.
   - **Dataset II**: Dr. Badawi’s Hand Vein Dataset, consisting of 500 images from 50 individuals.

2. **Preprocessing Pipeline**: A two-stage preprocessing pipeline is implemented, with a significant role played by a U-Net model. This model enhances vein segmentation, which is crucial for improving feature extraction and recognition accuracy.

3. **Model Performance**: The experimental results indicate that the ensemble model combining MobileNet and VGG16 outperforms individual models. The accuracy achieved is:
   - **98.61% for Dataset I**
   - **100% for Dataset II**

4. **Evaluation Metrics**: The model's performance is furt

INFO: HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO: HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"



Answer:
The research paper presents a dorsal hand vein recognition system that utilizes two publicly available datasets to evaluate its effectiveness. The key findings and results of the study are as follows:

1. **Datasets Used**: The study employs two datasets:
   - **Dataset I**: The Dorsal Hand Veins Image Database, which includes 1,104 images from 138 individuals.
   - **Dataset II**: Dr. Badawi’s Hand Vein Dataset, consisting of 500 images from 50 individuals.

2. **Preprocessing Pipeline**: A two-stage preprocessing pipeline is implemented, with a significant role played by a U-Net model. This model enhances vein segmentation, which is crucial for improving feature extraction and recognition accuracy.

3. **Model Performance**: The experimental results indicate that the ensemble model combining MobileNet and VGG16 outperforms individual models. The accuracy achieved is:
   - **98.61% for Dataset I**
   - **100% for Dataset II**

4. **Evaluation Metrics**: The model's performanc

# Evalution

In [ ]:

evaluation_data = [
    # 🔹 Basic concept
    {
        "query": "What biometric trait is used in this system?", 
        "ground_truth": ["dorsal hand vein", "hand vein"]  
    },
    {
        "query": "Why are dorsal hand veins used for recognition?", 
        "ground_truth": ["unique", "internal", "contactless"]  
    },
 
    # 🔹 Models
    {
        "query": "What CNN models were used for classification?", 
        "ground_truth": ["VGG16", "ResNet50", "MobileNet", "InceptionV3"] 
    },
    {
        "query": "Which segmentation model is used?", 
        "ground_truth": ["U-Net", "UNet"]  
    },
    {
        "query": "What backbone is used with U-Net?", 
        "ground_truth": ["ResNet50", "ResNet-50"]  # 2 variants
    },
 
    # 🔹 Ensemble
    {
        "query": "What ensemble models are used?", 
        "ground_truth": ["ResNet", "VGG16", "MobileNet"]  # Individual models (simpler!)
    },
    {
        "query": "Why ensemble models are used?", 
        "ground_truth": ["improve performance", "combine"]  # Simplified
    },
 
    # 🔹 Dataset
    {
        "query": "Which datasets are used in the study?", 
        "ground_truth": ["Dorsal Hand Vein", "Badawi"]  # Shortened
    },
    {
        "query": "How many images are in dataset II?", 
        "ground_truth": ["500"]  # Just the number
    },
 
    # 🔹 Preprocessing
    {
        "query": "What preprocessing techniques are used?", 
        "ground_truth": ["CLAHE", "threshold", "morphological", "Canny"]  # Simplified terms
    },
    {
        "query": "What is the role of CLAHE?", 
        "ground_truth": ["contrast enhancement", "contrast"]  # 2 variants
    },
 
    # 🔹 Training
    {
        "query": "What optimizer is used?", 
        "ground_truth": ["Adam"]  # Single item
    },
    {
        "query": "What loss function is used?", 
        "ground_truth": ["cross entropy", "categorical cross entropy"]  # 2 key variants
    },
    {
        "query": "What activation functions are used?", 
        "ground_truth": ["ReLU", "Softmax"]  # Keep original
    },
 
    # 🔹 Performance
    {
        "query": "How much accuracy improvement is achieved using U-Net?", 
        "ground_truth": ["17%", "17"]  # 2 variants
    },
    {
        "query": "What is the best performing model?", 
        "ground_truth": ["MobileNet", "VGG16", "MobileNet+VGG"]  # Key components
    },
 
    # 🔹 Metrics
    {
        "query": "What evaluation metrics are used?", 
        "ground_truth": ["precision", "recall", "F1", "accuracy"]  # Keep original
    },
 
    # 🔹 Architecture
    {
        "query": "What are the stages of the system?", 
        "ground_truth": ["preprocessing", "segmentation", "classification"]  # Keep original
    },
 
    # 🔹 General
    {
        "query": "What is the purpose of dorsal vein recognition?", 
        "ground_truth": ["identification", "authentication"]  # Simplified
    },
    {
        "query": "Why is vein biometrics secure?", 
        "ground_truth": ["internal", "difficult to replicate"]  # Keep original
    },
]
 

In [33]:
"""
Simple RAG Pipeline Evaluation
================================
Clean, easy-to-understand evaluation code
"""

import numpy as np
import json
import time

# ============================================================================
# Simple Evaluation Function
# ============================================================================

def evaluate_retrieval(docs, ground_truth, k=5):
    """Check how many ground truth items found in top-k docs"""
    found = set()
    
    for doc in docs[:k]:
        # Check both AI summary and raw text
        text = doc.page_content.lower()
        
        # Also check original raw text
        if "original_content" in doc.metadata:
            try:
                original = json.loads(doc.metadata["original_content"])
                raw_text = original.get("raw_text", "")
                text = text + " " + raw_text.lower()
            except:
                pass
        
        # Check which ground truth items are present
        for gt_item in ground_truth:
            if gt_item.lower() in text:
                found.add(gt_item)
    
    # Calculate metrics
    precision = len(found) / k
    recall = len(found) / len(ground_truth)
    
    return precision, recall, len(found)


# ============================================================================
# Run Evaluation
# ============================================================================

print("="*70)
print("🔍 RAG PIPELINE EVALUATION")
print("="*70)

results = []

for idx, item in enumerate(evaluation_data):
    query = item["query"]
    gt = item["ground_truth"]
    
    print(f"\n[{idx+1}/{len(evaluation_data)}] {query[:60]}...")
    
    # Get query embedding
    query_vec = np.array(embeddings.embed_query(query))
    
    # Retrieve documents
    retrieved = hybrid.search(query, k=10, query_vec=query_vec)
    
    # Evaluate top-5 WITHOUT reranking
    p, r, found = evaluate_retrieval(retrieved[:5], gt, k=5)
    
    print(f"  P@5={p:.2f} | R@5={r:.2f} | Found {found}/{len(gt)}")
    
    results.append({
        "query": query,
        "precision": p,
        "recall": r,
        "found": found,
        "total": len(gt)
    })

# ============================================================================
# Print Summary
# ============================================================================

print("\n" + "="*70)
print("📊 RESULTS")
print("="*70)

avg_precision = np.mean([r["precision"] for r in results])
avg_recall = np.mean([r["recall"] for r in results])

print(f"\nAverage Precision@5: {avg_precision:.3f}")
print(f"Average Recall@5:    {avg_recall:.3f}")

print("\n" + "="*70)

🔍 RAG PIPELINE EVALUATION

[1/20] What biometric trait is used in this system?...


INFO: HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


  P@5=0.40 | R@5=1.00 | Found 2/2

[2/20] Why are dorsal hand veins used for recognition?...


INFO: HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


  P@5=0.40 | R@5=0.67 | Found 2/3

[3/20] What CNN models were used for classification?...


...

  P@5=0.60 | R@5=1.00 | Found 3/3

[19/20] What is the purpose of dorsal vein recognition?...


INFO: HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


  P@5=0.40 | R@5=1.00 | Found 2/2

[20/20] Why is vein biometrics secure?...


INFO: HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


  P@5=0.40 | R@5=1.00 | Found 2/2

📊 RESULTS

Average Precision@5: 0.420
Average Recall@5:    0.867



In [34]:
"""
Reranking Evaluation - Precision@5 and Recall@5
=================================================
Compares top-5 from hybrid vs top-5 from reranking
"""
 
import numpy as np
import json
import time
 
# ============================================================================
# CONFIG
# ============================================================================
 
TEST_QUERIES = 8  # How many queries to test
DELAY_SECONDS = 15
 
# ============================================================================
# Evaluation Function
# ============================================================================
 
def evaluate_retrieval(docs, ground_truth, k=5):
    """Evaluate precision and recall at k"""
    found = set()
    
    for doc in docs[:k]:
        text = doc.page_content.lower()
        
        if "original_content" in doc.metadata:
            try:
                original = json.loads(doc.metadata["original_content"])
                text += " " + original.get("raw_text", "").lower()
            except:
                pass
        
        for gt_item in ground_truth:
            if gt_item.lower() in text:
                found.add(gt_item)
    
    precision = len(found) / k if k > 0 else 0
    recall = len(found) / len(ground_truth) if len(ground_truth) > 0 else 0
    
    return precision, recall, len(found)
 
 
# ============================================================================
# Reranking Impact Test - P@5 and R@5
# ============================================================================
 
print("="*70)
print(f"🏆 RERANKING IMPACT TEST (Precision@5 & Recall@5)")
print("="*70)
print("Comparing: Hybrid top-5 vs Reranked top-5")
print("="*70)
 
results = []
subset = evaluation_data[:TEST_QUERIES]
 
for idx, item in enumerate(subset):
    query = item["query"]
    gt = item["ground_truth"]
    
    print(f"\n[{idx+1}/{TEST_QUERIES}] {query[:60]}...")
    
    # Step 1: Retrieve top-10 from hybrid
    query_vec = np.array(embeddings.embed_query(query))
    retrieved = hybrid.search(query, k=10, query_vec=query_vec)
    
    # Step 2: BEFORE reranking - evaluate top-5 from hybrid
    p_before, r_before, found_before = evaluate_retrieval(
        retrieved[:5],  # ← Top-5 from hybrid
        gt, 
        k=5
    )
    
    # Step 3: AFTER reranking - rerank and evaluate top-5
    try:
        reranked = llm_rerank(query, retrieved, top_k=5)  # Rerank 10 docs to top-5
        
        p_after, r_after, found_after = evaluate_retrieval(
            reranked,  # ← All 5 reranked docs (reranked[:5] is same as reranked)
            gt,
            k=5
        )
        
        # Calculate improvement
        improvement_p = p_after - p_before
        improvement_r = r_after - r_before
        
        print(f"  Before (hybrid top-5): P@5={p_before:.2f} | R@5={r_before:.2f} | Found {found_before}/{len(gt)}")
        print(f"  After  (rerank top-5): P@5={p_after:.2f} | R@5={r_after:.2f} | Found {found_after}/{len(gt)}")
        print(f"  Impact:                ΔP={improvement_p:+.2f} | ΔR={improvement_r:+.2f}")
        
        results.append({
            "query": query,
            "before": {"p": p_before, "r": r_before, "found": found_before},
            "after": {"p": p_after, "r": r_after, "found": found_after},
            "improvement": {"p": improvement_p, "r": improvement_r}
        })
        
        # Rate limit safety
        if idx < len(subset) - 1:
            print(f"  ⏳ Waiting {DELAY_SECONDS}s...")
            time.sleep(DELAY_SECONDS)
    
    except Exception as e:
        print(f"  ❌ Error: {e}")
        time.sleep(DELAY_SECONDS * 2)
        continue
 
 
# ============================================================================
# Summary
# ============================================================================
 
if results:
    print("\n" + "="*70)
    print("📊 RERANKING IMPACT SUMMARY (P@5 & R@5)")
    print("="*70)
    
    avg_p_before = np.mean([r["before"]["p"] for r in results])
    avg_r_before = np.mean([r["before"]["r"] for r in results])
    avg_p_after = np.mean([r["after"]["p"] for r in results])
    avg_r_after = np.mean([r["after"]["r"] for r in results])
    
    avg_improvement_p = np.mean([r["improvement"]["p"] for r in results])
    avg_improvement_r = np.mean([r["improvement"]["r"] for r in results])
    
    print(f"\n📍 Before Reranking (Hybrid Top-5):")
    print(f"  Precision@5: {avg_p_before:.3f}")
    print(f"  Recall@5:    {avg_r_before:.3f}")
    
    print(f"\n🏆 After Reranking (Top-5):")
    print(f"  Precision@5: {avg_p_after:.3f}")
    print(f"  Recall@5:    {avg_r_after:.3f}")
    
    print(f"\n📈 Improvement:")
    print(f"  ΔPrecision@5: {avg_improvement_p:+.3f}")
    print(f"  ΔRecall@5:    {avg_improvement_r:+.3f}")
    
    # Count improvements
    improved = sum(1 for r in results if r["improvement"]["p"] > 0 or r["improvement"]["r"] > 0)
    worse = sum(1 for r in results if r["improvement"]["p"] < 0 or r["improvement"]["r"] < 0)
    same = len(results) - improved - worse
    
    print(f"\n📊 Query Breakdown:")
    print(f"  Improved: {improved}/{len(results)} ({improved/len(results)*100:.0f}%)")
    print(f"  Worse:    {worse}/{len(results)} ({worse/len(results)*100:.0f}%)")
    print(f"  Same:     {same}/{len(results)} ({same/len(results)*100:.0f}%)")
    
    # Overall verdict
    if avg_improvement_p > 0.05 or avg_improvement_r > 0.05:
        print(f"\n✅ VERDICT: Reranking is helping significantly!")
        print(f"   Recommendation: Keep reranking in production")
    elif avg_improvement_p > 0 or avg_improvement_r > 0:
        print(f"\n✓ VERDICT: Reranking provides small improvement")
        print(f"   Recommendation: Keep if cost/latency acceptable")
    else:
        print(f"\n⚠️ VERDICT: Reranking not providing benefit")
        print(f"   Recommendation: Skip reranking, use hybrid directly")
    
    print("="*70)
 
else:
    print("\n⚠️ No results - all queries failed!")
 

🏆 RERANKING IMPACT TEST (Precision@5 & Recall@5)
Comparing: Hybrid top-5 vs Reranked top-5

[1/8] What biometric trait is used in this system?...


INFO: HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO: HTTP Request: POST https://api.cohere.com/v1/rerank "HTTP/1.1 200 OK"


  Before (hybrid top-5): P@5=0.40 | R@5=1.00 | Found 2/2
  After  (rerank top-5): P@5=0.40 | R@5=1.00 | Found 2/2
  Impact:                ΔP=+0.00 | ΔR=+0.00
  ⏳ Waiting 15s...

[2/8] Why are dorsal hand veins used for recognition?...


INFO: HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO: HTTP Request: POST https://api.cohere.com/v1/rerank "HTTP/1.1 200 OK"


  Before (hybrid top-5): P@5=0.40 | R@5=0.67 | Found 2/3
  After  (rerank top-5): P@5=0.60 | R@5=1.00 | Found 3/3
  Impact:                ΔP=+0.20 | ΔR=+0.33
  ⏳ Waiting 15s...

[3/8] What CNN models were used for classification?...


...

  Before (hybrid top-5): P@5=0.60 | R@5=1.00 | Found 3/3
  After  (rerank top-5): P@5=0.60 | R@5=1.00 | Found 3/3
  Impact:                ΔP=+0.00 | ΔR=+0.00
  ⏳ Waiting 15s...

[7/8] Why ensemble models are used?...


INFO: HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO: HTTP Request: POST https://api.cohere.com/v1/rerank "HTTP/1.1 200 OK"


  Before (hybrid top-5): P@5=0.20 | R@5=0.50 | Found 1/2
  After  (rerank top-5): P@5=0.40 | R@5=1.00 | Found 2/2
  Impact:                ΔP=+0.20 | ΔR=+0.50
  ⏳ Waiting 15s...

[8/8] Which datasets are used in the study?...


INFO: HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO: HTTP Request: POST https://api.cohere.com/v1/rerank "HTTP/1.1 200 OK"


  Before (hybrid top-5): P@5=0.40 | R@5=1.00 | Found 2/2
  After  (rerank top-5): P@5=0.40 | R@5=1.00 | Found 2/2
  Impact:                ΔP=+0.00 | ΔR=+0.00

📊 RERANKING IMPACT SUMMARY (P@5 & R@5)

📍 Before Reranking (Hybrid Top-5):
  Precision@5: 0.425
  Recall@5:    0.833

🏆 After Reranking (Top-5):
  Precision@5: 0.475
  Recall@5:    0.938

📈 Improvement:
  ΔPrecision@5: +0.050
  ΔRecall@5:    +0.104

📊 Query Breakdown:
  Improved: 2/8 (25%)
  Worse:    0/8 (0%)
  Same:     6/8 (75%)

✅ VERDICT: Reranking is helping significantly!
   Recommendation: Keep reranking in production


In [ ]:
"""
Complete Evaluation: Retrieval + Reranking + LLM Answers
==========================================================
Shows P@5, R@5 metrics + actual LLM answers
"""

import numpy as np
import json
import time

# ============================================================================
# CONFIG
# ============================================================================

TEST_QUERIES = 8  # How many queries to test
DELAY_SECONDS = 15
SHOW_FULL_ANSWER = True  # Set False to show truncated answers

# ============================================================================
# Evaluation Function
# ============================================================================

def evaluate_retrieval(docs, ground_truth, k=5):
    """Evaluate precision and recall at k"""
    found = set()
    
    for doc in docs[:k]:
        text = doc.page_content.lower()
        
        if "original_content" in doc.metadata:
            try:
                original = json.loads(doc.metadata["original_content"])
                text += " " + original.get("raw_text", "").lower()
            except:
                pass
        
        for gt_item in ground_truth:
            if gt_item.lower() in text:
                found.add(gt_item)
    
    precision = len(found) / k if k > 0 else 0
    recall = len(found) / len(ground_truth) if len(ground_truth) > 0 else 0
    
    return precision, recall, len(found)


# ============================================================================
# Complete Evaluation with LLM Answers
# ============================================================================

print("="*70)
print(f"🎯 COMPLETE RAG EVALUATION (Retrieval + Reranking + Answers)")
print("="*70)
print(f"Testing {TEST_QUERIES} queries with full pipeline")
print("="*70)

results = []
subset = evaluation_data[:TEST_QUERIES]

for idx, item in enumerate(subset):
    query = item["query"]
    gt = item["ground_truth"]
    
    print(f"\n{'='*70}")
    print(f"[{idx+1}/{TEST_QUERIES}] QUERY: {query}")
    print(f"{'='*70}")
    print(f"Ground Truth Items: {gt}")
    print(f"{'-'*70}")
    
    # Step 1: Retrieve from hybrid
    query_vec = np.array(embeddings.embed_query(query))
    retrieved = hybrid.search(query, k=10, query_vec=query_vec)
    
    # Step 2: Evaluate BEFORE reranking
    p_before, r_before, found_before = evaluate_retrieval(retrieved[:5], gt, k=5)
    print(f"\n📊 BEFORE Reranking (Hybrid Top-5):")
    print(f"   Precision@5: {p_before:.2f} | Recall@5: {r_before:.2f} | Found: {found_before}/{len(gt)}")
    
    # Step 3: Rerank
    try:
        reranked = llm_rerank(query, retrieved, top_k=5)
        
        # Step 4: Evaluate AFTER reranking
        p_after, r_after, found_after = evaluate_retrieval(reranked, gt, k=5)
        print(f"\n🏆 AFTER Reranking (Top-5):")
        print(f"   Precision@5: {p_after:.2f} | Recall@5: {r_after:.2f} | Found: {found_after}/{len(gt)}")
        
        # Step 5: Calculate improvement
        improvement_p = p_after - p_before
        improvement_r = r_after - r_before
        print(f"\n📈 Impact: ΔP={improvement_p:+.2f} | ΔR={improvement_r:+.2f}")
        
        # Step 6: Generate LLM Answer using reranked docs
        print(f"\n💬 GENERATING LLM ANSWER...")
        print(f"{'-'*70}")
        
        answer = ""
        try:
            # Generate answer with reranked docs
            for chunk in generate_final_answer_stream(reranked, query):
                if SHOW_FULL_ANSWER:
                    print(chunk, end="", flush=True)
                answer += chunk
            
            print()  # Newline after streaming
            
            if not SHOW_FULL_ANSWER:
                # Show truncated answer
                print(answer[:300] + "..." if len(answer) > 300 else answer)
        
        except Exception as e:
            answer = f"[Error generating answer: {e}]"
            print(f"❌ {answer}")
        
        # Store results
        results.append({
            "query": query,
            "ground_truth": gt,
            "before": {"p": p_before, "r": r_before, "found": found_before},
            "after": {"p": p_after, "r": r_after, "found": found_after},
            "improvement": {"p": improvement_p, "r": improvement_r},
            "answer": answer
        })
        
        # Rate limit safety
        if idx < len(subset) - 1:
            print(f"\n⏳ Waiting {DELAY_SECONDS}s for rate limit...")
            time.sleep(DELAY_SECONDS)
    
    except Exception as e:
        print(f"❌ Error in reranking/answer generation: {e}")
        time.sleep(DELAY_SECONDS * 2)
        continue


# ============================================================================
# Summary Statistics
# ============================================================================

if results:
    print("\n" + "="*70)
    print(" FINAL SUMMARY")
    print("="*70)
    
    # Retrieval metrics
    avg_p_before = np.mean([r["before"]["p"] for r in results])
    avg_r_before = np.mean([r["before"]["r"] for r in results])
    avg_p_after = np.mean([r["after"]["p"] for r in results])
    avg_r_after = np.mean([r["after"]["r"] for r in results])
    avg_improvement_p = np.mean([r["improvement"]["p"] for r in results])
    avg_improvement_r = np.mean([r["improvement"]["r"] for r in results])
    
    print(f"\  Hybrid Retrieval (Top-5):")
    print(f"   Average Precision@5: {avg_p_before:.3f}")
    print(f"   Average Recall@5:    {avg_r_before:.3f}")
    
    print(f"\n With Reranking (Top-5):")
    print(f"   Average Precision@5: {avg_p_after:.3f}")
    print(f"   Average Recall@5:    {avg_r_after:.3f}")
    
    print(f"\n Reranking Impact:")
    print(f"   ΔPrecision@5: {avg_improvement_p:+.3f}")
    print(f"   ΔRecall@5:    {avg_improvement_r:+.3f}")
    
    # Count improvements
    improved = sum(1 for r in results if r["improvement"]["p"] > 0 or r["improvement"]["r"] > 0)
    worse = sum(1 for r in results if r["improvement"]["p"] < 0 or r["improvement"]["r"] < 0)
    same = len(results) - improved - worse
    
    print(f"\n Query Breakdown:")
    print(f"   Improved: {improved}/{len(results)} ({improved/len(results)*100:.0f}%)")
    print(f"   Worse:    {worse}/{len(results)} ({worse/len(results)*100:.0f}%)")
    print(f"   Same:     {same}/{len(results)} ({same/len(results)*100:.0f}%)")
    
    # Answer quality (manual inspection needed)
    print(f"\n Answer Quality:")
    print(f"   Total answers generated: {len([r for r in results if r['answer']])}")
    print(f"   → Manual review needed to assess answer quality")
    print(f"   → Scroll up to see individual answers")
    
    print("="*70)

else:
    print("\n No results - all queries failed!")


# ============================================================================
# Save Results to File
# ============================================================================

# Save to JSON for later review
output_file = "evaluation_results_with_answers.json"

try:
    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(results, f, indent=2, ensure_ascii=False)
    print(f"\n Results saved to: {output_file}")
    print(f"   You can review all answers later in this file")
except Exception as e:
    print(f"\n Could not save results: {e}")


# ============================================================================
# Quick Answer Review
# ============================================================================

print("\n" + "="*70)
print("QUICK ANSWER REVIEW")
print("="*70)

for idx, r in enumerate(results, 1):
    print(f"\n[{idx}] {r['query']}")
    answer_preview = r['answer'][:150] + "..." if len(r['answer']) > 150 else r['answer']
    print(f"    Answer: {answer_preview}")
    print(f"    Metrics: P@5={r['after']['p']:.2f}, R@5={r['after']['r']:.2f}")

print("\n" + "="*70)

🎯 COMPLETE RAG EVALUATION (Retrieval + Reranking + Answers)
Testing 8 queries with full pipeline

[1/8] QUERY: What biometric trait is used in this system?
Ground Truth Items: ['dorsal hand vein', 'hand vein']
----------------------------------------------------------------------


INFO: HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"



📊 BEFORE Reranking (Hybrid Top-5):
   Precision@5: 0.40 | Recall@5: 1.00 | Found: 2/2


INFO: HTTP Request: POST https://api.cohere.com/v1/rerank "HTTP/1.1 200 OK"



🏆 AFTER Reranking (Top-5):
   Precision@5: 0.40 | Recall@5: 1.00 | Found: 2/2

📈 Impact: ΔP=+0.00 | ΔR=+0.00

💬 GENERATING LLM ANSWER...
----------------------------------------------------------------------


...


📊 BEFORE Reranking (Hybrid Top-5):
   Precision@5: 0.40 | Recall@5: 1.00 | Found: 2/2


INFO: HTTP Request: POST https://api.cohere.com/v1/rerank "HTTP/1.1 200 OK"



🏆 AFTER Reranking (Top-5):
   Precision@5: 0.40 | Recall@5: 1.00 | Found: 2/2

📈 Impact: ΔP=+0.00 | ΔR=+0.00

💬 GENERATING LLM ANSWER...
----------------------------------------------------------------------


INFO: HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


The study utilizes two datasets for hand vein recognition:

1. **Dorsal Hand Veins Image Database**: This dataset consists of two parts:
   - **Database 1**: 138 individuals, with four images per person per hand, totaling 1,104 images.
   - **Database 2**: 113 individuals, with three images per person per hand, totaling 678 images.

2. **Dr. Badawi’s Hand Vein Dataset**: This dataset includes 500 images from 50 individuals, with five images per person per hand.

These datasets are used to enhance the accuracy of vein pattern recognition through a two-stage preprocessing pipeline.

📊 FINAL SUMMARY

📍 Hybrid Retrieval (Top-5):
   Average Precision@5: 0.425
   Average Recall@5:    0.833

🏆 With Reranking (Top-5):
   Average Precision@5: 0.475
   Average Recall@5:    0.938

📈 Reranking Impact:
   ΔPrecision@5: +0.050
   ΔRecall@5:    +0.104

📊 Query Breakdown:
   Improved: 2/8 (25%)
   Worse:    0/8 (0%)
   Same:     6/8 (75%)

💬 Answer Quality:
   Total answers generated: 8
   → Manual re